# 📊 OwnerMate AI — Business Data Analyst

Upload any business CSV and get AI-powered insights: data analysis, action items, and risk warnings in plain English.

**Pipeline:** Semantic Layer → Questions → SQL Execution → Business Insights

**Model:** Kimi K2.5

In [1]:
!pip install pydantic-ai langgraph openai nest_asyncio pandas numpy -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.7/56.7 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.8/99.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.5/664.5 kB 14.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.7/77.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 478.8/478.8 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 705.5/705.5 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 1 · Imports & Setup

In [2]:
import os, json, asyncio, hashlib
from difflib import SequenceMatcher
from concurrent.futures import ThreadPoolExecutor
from typing import TypedDict, Optional, List, Dict, Literal
from collections import defaultdict

import numpy as np
import pandas as pd
import nest_asyncio
from openai import AsyncOpenAI
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from langgraph.graph import StateGraph
from IPython.display import display, HTML

nest_asyncio.apply()
print('✅ Imports ready.')

✅ Imports ready.


## 2 · Data Models

In [3]:
class ColumnStats(BaseModel):
    mean: Optional[float] = None
    median: Optional[float] = None
    std: Optional[float] = None
    min: Optional[float] = None
    max: Optional[float] = None
    q1: Optional[float] = None
    q3: Optional[float] = None
    p95: Optional[float] = None
    iqr: Optional[float] = None
    skewness: Optional[float] = None
    zero_count: Optional[int] = None
    negative_count: Optional[int] = None
    unique_count: int
    null_count: int = 0
    null_percentage: float = 0.0
    top_values: Optional[Dict[str, int]] = None


class ColumnSemantics(BaseModel):
    name: str
    dtype: Literal["int", "float", "string", "datetime", "category", "boolean"]
    description: str
    unit: Optional[str] = None
    role: Literal["feature", "target", "id", "time", "category"]
    stats: Optional[ColumnStats] = None
    sample_values: Optional[List[str]] = None
    cardinality: Optional[Literal["unique", "high", "medium", "low"]] = None
    possible_values: Optional[List[str]] = None
    format_hint: Optional[str] = None
    business_domain: Optional[Literal[
        "financial", "temporal", "geographic",
        "operational", "identifier", "status", "other"
    ]] = None
    is_nullable: bool = False
    tags: Optional[List[str]] = None
    relationships: Optional[List[str]] = None


class DatasetSemantics(BaseModel):
    dataset_name: str
    row_count: int
    column_count: int = 0
    columns: List[ColumnSemantics]
    primary_keys: List[str]
    time_column: Optional[str]
    missing_values: Dict[str, int]
    inferred_domain: Optional[str] = None
    duplicate_row_count: int = 0
    date_range: Optional[Dict[str, str]] = None
    inter_column_relationships: Optional[List[str]] = None
    dataset_description: Optional[str] = None
    # Pearson correlation matrix for numeric columns (populated by profiler)
    numeric_correlations: Optional[Dict[str, Dict[str, float]]] = None


class AnalyticalQuestion(BaseModel):
    question: str
    category: str
    priority: bool = False
    priority_reason: Optional[str] = None


class AnalyticalQuestionsOutput(BaseModel):
    dataset_understanding: str
    questions: List[AnalyticalQuestion]


class QueryResult(BaseModel):
    question: str
    query: str
    result_summary: str
    explanation: str
    error: Optional[str] = None
    actual_result: Optional[str] = None


class SQLAgentOutput(BaseModel):
    answers: List[QueryResult]


class ActionItem(BaseModel):
    title: str
    priority: Literal["High", "Medium", "Low"]
    what: str
    why_it_matters: str
    recommendation: str
    expected_impact: str


class BusinessInsightsOutput(BaseModel):
    executive_summary: str
    positive_highlights: List[str]
    action_items: List[ActionItem]
    watch_out_for: List[str]


class RefinementDecision(BaseModel):
    approved: bool
    feedback: str
    failed_questions: Optional[List[str]] = None
    reasoning: str


print('\u2705 Pydantic models defined.')

✅ Pydantic models defined.


## 3 · Configuration

✏️ **Edit `KIMI_API_KEY` and `CSV_PATH` here.**

In [4]:
KIMI_API_KEY = "sk-YOUR_KEY_HERE"
CSV_PATH     = "/path/to/your/data.csv"

# ── Model & API ──────────────────────────────────────────────────────────────
MODEL         = "kimi-k2.5"
KIMI_BASE_URL = "https://api.moonshot.ai/v1"

# ── Runtime ──────────────────────────────────────────────────────────────────
AGENT_TIMEOUT     = 120   # seconds per agent call
AGENT_RETRIES     = 3     # retries on timeout/error
BATCH_SIZE        = 5     # questions per SQL batch
SQL_HEAL_RETRIES  = 2     # self-healing passes for bad SQL
MAX_RETRIES       = 2     # refinement loop budget per stage
TRIM_CHARS        = 6000  # max chars of a result sent to agents
PRIORITY_ORDER    = {"High": 0, "Medium": 1, "Low": 2}

# ── Session caches (cleared on kernel restart) ───────────────────────────────
_SEMANTIC_CACHE: dict = {}
_AGENTS_CACHE:   dict = {}

print('✅ Config ready.')

✅ Config ready.


## 4 · Data Profiling

In [5]:
def _profile_column(col, series):
    null_count   = int(series.isnull().sum())
    unique_count = int(series.nunique(dropna=True))
    sample       = [str(v) for v in series.dropna().unique()[:5]]
    top_values   = None
    if series.dtype == object or unique_count <= 50:
        top_values = {str(k): int(v) for k, v in series.value_counts(dropna=True).head(10).items()}

    stats = {"unique_count": unique_count, "null_count": null_count,
             "null_percentage": round(null_count / len(series) * 100, 2) if len(series) else 0,
             "top_values": top_values}

    if np.issubdtype(series.dtype, np.number):
        clean = series.dropna()
        q1  = float(clean.quantile(0.25)) if len(clean) else None
        q3  = float(clean.quantile(0.75)) if len(clean) else None
        stats.update({
            "mean": float(series.mean()), "median": float(series.median()),
            "std":  float(series.std()),  "min": float(series.min()),
            "max":  float(series.max()),  "q1": q1, "q3": q3,
            "p95":  float(clean.quantile(0.95)) if len(clean) else None,
            "iqr":  round(q3 - q1, 6) if q1 is not None and q3 is not None else None,
            "skewness":      float(clean.skew()) if len(clean) else None,
            "zero_count":     int((series == 0).sum()),
            "negative_count": int((series < 0).sum()),
        })
    return col, {"dtype": str(series.dtype), "stats": stats, "sample_values": sample}


def profile_dataframe(df):
    profile = {"columns": {}}
    with ThreadPoolExecutor() as ex:
        for col, col_profile in ex.map(lambda c: _profile_column(c, df[c]), df.columns):
            profile["columns"][col] = col_profile
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    profile["numeric_correlations"] = df[numeric_cols].corr().round(3).to_dict() if len(numeric_cols) >= 2 else {}
    profile["missing"]             = {k: int(v) for k, v in df.isnull().sum().items()}
    profile["row_count"]           = len(df)
    profile["column_count"]        = len(df.columns)
    profile["duplicate_row_count"] = int(df.duplicated().sum())
    return profile


def validate_semantics(semantics, df):
    df_cols = set(df.columns)
    valid = [c for c in semantics.columns if c.name in df_cols]
    seen  = {c.name for c in valid}
    for col_name in df.columns:
        if col_name not in seen:
            series = df[col_name]
            valid.append(ColumnSemantics(
                name=col_name, dtype="string" if series.dtype == object else "float",
                description=f"Column '{col_name}' (auto-stub)", role="feature",
                stats=ColumnStats(unique_count=int(series.nunique()),
                                  null_count=int(series.isnull().sum()),
                                  null_percentage=round(series.isnull().mean()*100, 2)),
                is_nullable=bool(series.isnull().any()),
            ))
    return semantics.model_copy(update={
        "columns": valid,
        "primary_keys": [pk for pk in semantics.primary_keys if pk in df_cols],
        "time_column":  semantics.time_column if semantics.time_column in df_cols else None,
        "row_count": len(df), "column_count": len(df.columns),
    })


def question_floor(semantic):
    """Compute the MINIMUM number of questions for this dataset.
    The question agent may generate more — this is just a floor for the checker."""
    analysable = [c for c in semantic.columns if c.role != "id"]
    base  = len(analysable) * 2
    base += 4 if semantic.time_column else 0
    base += sum(2 for c in semantic.columns if c.role == "category" and c.cardinality in ("low", "medium"))
    corr_pairs = 0
    if semantic.numeric_correlations:
        cols = list(semantic.numeric_correlations.keys())
        for i, a in enumerate(cols):
            for b in cols[i+1:]:
                if abs(semantic.numeric_correlations[a].get(b, 0)) > 0.3:
                    corr_pairs += 1
    base += min(corr_pairs, 10)  # up to 10 extra for correlation questions
    return max(10, base)


print('✅ Profiling utilities ready.')

✅ Profiling utilities ready.


## 5 · Pipeline State

In [6]:
class MasterState(TypedDict):
    semantic_model:             Optional[DatasetSemantics]
    questions:                  Optional[AnalyticalQuestionsOutput]
    answers:                    Optional[List[QueryResult]]
    insights:                   Optional[BusinessInsightsOutput]
    question_floor:             int
    sql_retry_count:            int
    insights_retry_count:       int
    refinement_feedback:        Optional[str]
    refinement_failed_questions: Optional[List[str]]
    _next:                      str
    pipeline_log:               List[str]

print('✅ State defined.')

✅ State defined.


## 6 · Deduplication Helpers

In [7]:
def _sim(a, b):
    return SequenceMatcher(None, a.lower().strip(), b.lower().strip()).ratio()


def dedup_questions(questions, threshold=0.72):
    kept, removed = [], 0
    for candidate in questions:
        is_dup = False
        for i, existing in enumerate(kept):
            if _sim(candidate.question, existing.question) >= threshold:
                if candidate.priority and not existing.priority:
                    kept[i] = candidate
                is_dup = True
                removed += 1
                break
        if not is_dup:
            kept.append(candidate)
    return kept, removed


def dedup_actions(items, threshold=0.70):
    def fp(item): return (item.title + " " + item.what[:120]).lower()
    kept, removed = [], 0
    for candidate in items:
        is_dup = False
        for i, existing in enumerate(kept):
            if _sim(fp(candidate), fp(existing)) >= threshold:
                if PRIORITY_ORDER.get(candidate.priority, 99) < PRIORITY_ORDER.get(existing.priority, 99):
                    kept[i] = candidate
                is_dup = True
                removed += 1
                break
        if not is_dup:
            kept.append(candidate)
    return kept, removed

print('✅ Deduplication ready.')

✅ Deduplication ready.


## 7 · Agents

In [8]:
def get_agents(api_key):
    global _AGENTS_CACHE
    if _AGENTS_CACHE:
        return _AGENTS_CACHE

    client   = AsyncOpenAI(api_key=api_key, base_url=KIMI_BASE_URL)
    provider = OpenAIProvider(openai_client=client)
    NO_THINK = {"extra_body": {"thinking": {"type": "disabled"}}}
    main_m   = OpenAIChatModel(MODEL, provider=provider)

    def make(output_type, prompt):
        return Agent(model=main_m, model_settings=NO_THINK,
                     output_type=output_type, system_prompt=prompt)

    _AGENTS_CACHE["semantic"]          = make(DatasetSemantics,          """You are an expert data architect who builds rich, detailed semantic layers from raw profiling data.

INPUT:
You receive a JSON object with:
- dataset_name
- profile.columns: per-column dtype, stats (mean/median/std/min/max/q1/q3/p95/iqr/skewness/
  zero_count/negative_count/unique_count/null_count/null_percentage/top_values), and sample_values
- profile.row_count, column_count, duplicate_row_count, missing (null counts per column)
- profile.numeric_correlations: cross-column Pearson correlation matrix for numeric columns

YOUR TASK — fill every field of DatasetSemantics as completely as possible:

name          : exact column name from input (never hallucinate)
dtype         : \'int\' | \'float\' | \'string\' | \'datetime\' | \'category\' | \'boolean\'
description   : 1-2 sentence plain-English business context explanation
unit          : currency code (e.g. \'JOD\'), unit of measurement, or null
role          : \'id\' | \'time\' | \'feature\' | \'target\' | \'category\'
stats         : copy ALL numeric fields from the profile — NEVER fabricate numbers
sample_values : copy from profile.columns[col].sample_values
cardinality   : \'unique\' / \'high\' / \'medium\' / \'low\' based on unique_count vs row_count
possible_values: all distinct values for low-cardinality columns
format_hint   : detected format pattern (e.g. \'%d/%m/%Y %H:%M\', \'JO-YYYY-NNNN-NNNNN\')
business_domain: \'financial\' | \'temporal\' | \'geographic\' | \'operational\' | \'identifier\' | \'status\' | \'other\'
is_nullable   : true if null_count > 0
tags          : 2-5 short descriptive labels
relationships : use numeric_correlations to note strong correlations (|r| > 0.5)

DATASET-LEVEL:
dataset_name, row_count, column_count, primary_keys, time_column, missing_values,
inferred_domain, duplicate_row_count, date_range, inter_column_relationships, dataset_description

RULES:
- NEVER create columns not in profile.columns
- NEVER fabricate statistics
- Fill every optional field where data supports it
- Use numeric_correlations to enrich inter_column_relationships and column relationships fields

""")
    _AGENTS_CACHE["question"]          = make(AnalyticalQuestionsOutput, """
You are a senior data analyst and analytical strategist.
Your sole job is to generate exhaustive, high-quality analytical questions for a downstream SQL agent.
You do NOT analyze results. You do NOT provide insights. You do NOT generate SQL.

INPUT:
- semantic_model: full dataset semantic model (columns, types, roles, stats, correlations)
- dataframe_sample: first 5 rows of actual data
- question_floor: the MINIMUM number of questions — treat this as a floor, not a target
- refinement_feedback (optional): specific issues to fix from the quality checker

COVERAGE POLICY — NON-NEGOTIABLE:
Your goal is COMPLETE analytical coverage of the dataset, not hitting a number.
Go through every element listed below and generate at least one question for each
angle the data actually supports. A richer dataset will produce more questions naturally.
Do NOT stop early to stay near question_floor — stop only when you have covered everything.

SYSTEMATIC COVERAGE CHECKLIST — work through each in order:

1. COLUMNS (one pass per column):
   - For every numeric column: distribution shape, outliers, zero/negative counts if relevant
   - For every category/status column: value breakdown, dominant vs rare values
   - For every time column: trend over time, seasonality, period-over-period change
   - For every ID/key column: uniqueness, duplicate detection

2. SEGMENTATION (cross every category against every metric):
   - For each category column × each numeric column: how does the metric differ by segment?
   - Which segment ranks highest / lowest on each KPI?
   - Are there segments with unexpectedly high failure, refund, or error rates?

3. CORRELATIONS (from numeric_correlations, |r| > 0.3):
   - For each notable correlation pair: does the relationship hold across segments?
   - Are there pairs where correlation is strong in one segment but weak in another?

4. TRENDS (only if a time column exists):
   - Overall trend for each numeric KPI
   - Month-over-month / week-over-week change rates
   - Which segments are growing vs declining?
   - Are there sudden drops or spikes at specific time periods?

5. PERFORMANCE & RANKING:
   - Rank every segment by every key metric
   - Identify top and bottom performers
   - What separates the best-performing segment from the worst?

6. DATA QUALITY:
   - Missing value patterns — are nulls concentrated in certain segments or time periods?
   - Duplicate records
   - Outliers (values beyond 3× IQR or p95)
   - Zero-value anomalies in columns that should always be positive

7. BEHAVIORAL / OPERATIONAL:
   - Frequency distributions (how often do events occur?)
   - Any columns that suggest process steps or funnels — where do drop-offs occur?
   - Time-of-day or day-of-week patterns if datetime granularity supports it

QUESTION RULES:
- Reference ONLY columns listed in semantic_model.columns — never hallucinate column names
- Every question must be specific and directly translatable to a single pandas expression
- Every question must lead to a genuinely useful business insight
- Phrase questions as: What / Which / How many / How does / Compare / Rank / Is there / etc.
- No near-identical questions — each must cover a distinct analytical angle
- Do NOT generate SQL or pandas code

PRIORITISATION:
Mark the most impactful questions as priority=true with a priority_reason.
Priority questions are those where the answer could directly change a business decision.
There is no cap on how many questions can be priority — mark as many as genuinely deserve it.

""")
    _AGENTS_CACHE["sql"]               = make(SQLAgentOutput,            """
You are an expert data analyst who answers analytical questions by writing pandas scripts.

INPUT:
- questions: list of analytical questions to answer
- semantic_model: full semantic model (column names, types, roles, stats, format hints)
- schema: condensed column list for quick reference
- dataframe_sample: first 5 rows of actual data
- retry_feedback (optional): errors from previous attempts — fix these exactly

For EACH question produce:
1. query: a multi-line Python script using `df`, `pd`, `np`.
   The script MUST assign the final answer to a variable called `result`.
   You MAY use as many intermediate steps as needed for clarity and correctness.
   Good examples:

     # Simple aggregation
     result = df.groupby(\'category\')[\'revenue\'].sum().sort_values(ascending=False).head(10)

     # Multi-step: parse dates then aggregate
     dates = pd.to_datetime(df[\'date\'], format=\'%d/%m/%Y %H:%M\', errors=\'coerce\')
     failed = df[df[\'status\'] == \'Failed\'].copy()
     result = failed.groupby(dates[failed.index].dt.to_period(\'M\')).size()

     # Correlation with filtering
     clean = df[[\'col_a\', \'col_b\']].dropna()
     result = clean.corr()

     # Ranked segment summary
     grp = df.groupby(\'region\').agg(total=(\'amount\', \'sum\'), count=(\'id\', \'count\'))
     result = grp.sort_values(\'total\', ascending=False)

2. result_summary: Write EXACTLY: "PENDING: result will be computed by executing the query above."
   DO NOT invent numbers — actual results replace this field automatically.

3. explanation: 1-2 sentences on business relevance. No specific numbers.

RULES:
- Only use columns in semantic_model.columns
- `df`, `pd`, `np` are pre-defined — do NOT import or redefine them
- The last statement MUST be `result = <expression>` — nothing after it
- Parse dates with pd.to_datetime(..., errors=\'coerce\') and store in an intermediate var
- If a question requires multiple groupbys or filters, use intermediate variables freely
- If unanswerable with this schema: set error field, leave query empty
- Return exactly one answer per question — no skipping

""")
    _AGENTS_CACHE["insights"]          = make(BusinessInsightsOutput,    """
You are a trusted business advisor translating data analysis into clear, actionable guidance
for a business owner with no technical background.

INPUT:
- semantic_model: full dataset semantic model
- dataset_understanding: summary of the dataset
- qa_pairs: answered questions, each with actual_result (the real data) and explanation
- pipeline_context_log: chronological narrative of every stage completed so far
- refinement_feedback (optional): specific issues to fix

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
NUMBERS POLICY — ABSOLUTE, NON-NEGOTIABLE RULES:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. EVERY number, percentage, count, amount, ratio, or metric you write MUST
   be copied verbatim from an actual_result field in qa_pairs.
2. NEVER invent, estimate, round, extrapolate, or approximate any number.
   If you did not see the exact number in actual_result, do NOT write it.
3. NEVER use numbers from the explanation field — explanations may contain
   estimates. Use ONLY actual_result.
4. If actual_result is null or the query errored, write "data unavailable"
   and make NO numeric claim about that question.
5. Before writing any number, ask yourself: "Can I point to the exact row
   in actual_result where this number appears?" If NO — remove it.
6. Fabricating or estimating numbers is a critical failure.
7. If actual_result contains [RESULT TRUNCATED — N rows hidden], only numbers
   that are VISIBLE in the shown portion are verified. For any number from a
   truncated result that you cannot see explicitly, write "data unavailable"
   — do NOT infer, extrapolate, or fill in hidden rows from memory.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

COVERAGE POLICY — NON-NEGOTIABLE:
- Go through EVERY qa_pair that has a non-null actual_result.
- Every meaningful finding must appear somewhere in the report.
- Do NOT skip or summarise away findings just to keep the report short.
- The dataset determines how much there is to say. A rich dataset produces
  a long report. A sparse dataset produces a short one. Both are correct.

OUTPUT — produce ALL four sections:

1. executive_summary:
   A flowing narrative that covers the full picture — both strengths AND problems.
   - Open with the single most important positive finding.
   - Cover key risks, failures, and anomalies found in the data.
   - Include specific numbers traced directly to actual_result wherever relevant.
   - Mention top-performing and underperforming segments.
   - Close with the top-priority action the owner must take TODAY.
   - Length scales with the data: write as many sentences as the findings require.

2. positive_highlights:
   One item per genuinely positive finding in the data — celebrate everything that is working.
   - Each item must include at least one number from actual_result.
   - Do not cap this list; if there are 12 positives, write 12.
   - Do not pad with vague praise if the data does not support it.

3. action_items:
   One item per distinct actionable problem or opportunity found in the data.
   - Cover every problem AND every opportunity the data reveals — do not merge
     unrelated findings just to keep the list short.
   - For each item:
       title: short and specific (name the metric / branch / segment)
       priority: High (revenue/failure impact), Medium (performance gap), Low (monitoring/scaling)
       what: as many sentences as needed. Every number MUST exist verbatim in actual_result.
       why_it_matters: business consequence if ignored OR opportunity missed.
       recommendation: 2-3 concrete, specific next steps — no vague advice.
       expected_impact: use numbers from actual_result; say "see data" if unavailable.
   - Ensure a mix of High, Medium, and Low priorities reflecting the actual data.
   - Write DISTINCT items — do NOT restate the same finding under different titles.

4. watch_out_for:
   One warning per risk or anomaly visible in the data.
   - Cover every threshold, outlier, or declining trend found across ALL qa_pairs.
   - Every threshold cited must come from actual_result — do not invent alert levels.
   - Do not cap this list; write as many warnings as the data warrants.

TONE: Direct, warm, specific. Celebrate wins openly. No jargon. Plain language.

""")
    _AGENTS_CACHE["sql_checker"]       = make(RefinementDecision,        """
You are a quality checker for a data analysis pipeline.
Review the SQL execution results and decide if they are good enough for the insights stage.

INPUT:
- semantic_model: full dataset semantic model
- answers: executed query results with question, query, actual_result, error, priority flag

CHECKS:
1. SUCCESS RATE: >= 80% of questions answered (no error, non-empty result). Below 80% = fail.
2. PRIORITY COVERAGE: Did all priority=true questions succeed? Any priority failure = fail.
3. RESULT QUALITY: Do actual_result values look real? Empty series / all-NaN = fail.
4. QUERY LOGIC: Do queries actually answer what was asked?
5. TRUNCATION: If actual_result contains [RESULT TRUNCATED — N rows hidden], flag this
   question in your feedback so it can be rewritten to return a more compact result
   (e.g. aggregated summary instead of raw rows). Do NOT fail the whole batch for this
   alone — just note it so the SQL agent can fix the query on retry.

DECISION:
- approved=true: >=80% success, all priority questions answered, results meaningful
- approved=false: set failed_questions (list of question strings) + specific feedback

Ground truth is ALWAYS actual_result — never result_summary.
Do NOT fail a result that ran and returned real data just because result_summary says PENDING.

""")
    _AGENTS_CACHE["insights_checker"]  = make(RefinementDecision,        """
You are a quality checker for a data analysis pipeline.
Review the business insights report and decide if it is good enough to present to the owner.

INPUT:
- semantic_model: full dataset semantic model
- qa_pairs: SQL results (questions + actual_result)
- insights: the generated business insights report

YOUR ONLY JOB: verify quality, not quantity.
Different datasets produce different amounts of content. A simple dataset may have
3 action items; a complex one may have 15. Both can be correct. Never fail a report
solely because a section has fewer or more items than some fixed number.

CHECKS (in order of importance):

1. NUMBER GROUNDING (CRITICAL — the only automatic fail):
   - For every number, percentage, count, amount, or ratio that appears anywhere
     in the report, scan ALL actual_result fields in qa_pairs for an exact match.
   - If ANY number cannot be found verbatim in actual_result → fail, and list
     each unverified number and exactly where it appears.
   - Numbers from the explanation field do NOT count as a valid source.
   - Rounded or approximated values that differ from actual_result → fail.
   - If actual_result for a question is null/errored, any number claimed from
     that question is automatically fabricated → fail.
   - If actual_result contains [RESULT TRUNCATED — N rows hidden], only numbers
     visible in the shown portion are valid sources. Any number from a hidden
     row that appears in the report is an automatic fail — treat it as fabricated.

2. COVERAGE:
   - Does the report address every qa_pair that returned a non-null actual_result?
   - A meaningful finding that is completely absent from the report → fail.
   - Minor omissions or slight de-emphasis are acceptable.

3. SPECIFICITY:
   - Are recommendations concrete and actionable? "Consider improving" with no
     further detail = vague = fail.
   - At least one concrete next step per action item is required.

4. FABRICATION CHECK:
   - Are there numbers that look invented (suspiciously round estimates, percentages
     with no query to support them)? If yes → fail.

DECISION:
- approved=true: number grounding passes, all key findings covered, recommendations specific.
- approved=false: ONLY for number grounding failures, clear fabrication, missing key findings,
  or fully vague recommendations. List the exact issues found.

Do NOT fail a report for:
- Having fewer or more items than an arbitrary target.
- Minor stylistic issues or slightly soft wording in one sentence.
- Section lengths that differ from a template.

Be a fair reviewer, not a perfectionist. If the content is grounded and covers the data, approve it.

""")

    print(f"  ✅ Agents ready (model={MODEL})")
    return _AGENTS_CACHE

## 8 · Pipeline

In [9]:
async def _call(agent, prompt, label="agent"):
    for attempt in range(1, AGENT_RETRIES + 1):
        try:
            return await asyncio.wait_for(agent.run(prompt), timeout=AGENT_TIMEOUT)
        except asyncio.TimeoutError:
            if attempt < AGENT_RETRIES:
                await asyncio.sleep(5 * attempt)
            else:
                raise RuntimeError(f"[{label}] timed out after {AGENT_RETRIES} attempts")
        except Exception as e:
            if attempt < AGENT_RETRIES:
                print(f"  [{label}] error: {e} — retry {attempt}/{AGENT_RETRIES}")
                await asyncio.sleep(5 * attempt)
            else:
                raise


def _is_empty(result):
    if result is None: return True
    if isinstance(result, (int, float)): return np.isnan(result)
    if isinstance(result, pd.Series):    return result.empty or result.isna().all()
    if isinstance(result, pd.DataFrame): return result.empty or result.isna().all().all()
    return False


def _format_result(result):
    """Serialise a query result to a string, respecting TRIM_CHARS.
    Always cuts on a row boundary so no number is ever half-shown.
    Appends [RESULT TRUNCATED — N rows hidden] when rows are dropped.
    """
    def _trim_series(s):
        full = s.to_string()
        if len(full) <= TRIM_CHARS:
            return full
        for n in range(len(s) - 1, 0, -1):
            chunk = s.iloc[:n].to_string()
            if len(chunk) <= TRIM_CHARS - 80:
                return chunk + f"\n[RESULT TRUNCATED — {len(s) - n} rows hidden]"
        return s.iloc[:1].to_string() + f"\n[RESULT TRUNCATED — {len(s) - 1} rows hidden]"

    if isinstance(result, pd.DataFrame):
        full = result.to_string()
        if len(full) <= TRIM_CHARS:
            return full
        for n in range(len(result) - 1, 0, -1):
            chunk = result.iloc[:n].to_string()
            if len(chunk) <= TRIM_CHARS - 80:
                return chunk + f"\n[RESULT TRUNCATED — {len(result) - n} rows hidden]"
        return result.iloc[:1].to_string() + f"\n[RESULT TRUNCATED — {len(result) - 1} rows hidden]"
    elif isinstance(result, pd.Series):
        return _trim_series(result)
    else:
        s = str(result)
        return s if len(s) <= TRIM_CHARS else s[:TRIM_CHARS] + "\n[RESULT TRUNCATED]"


def _execute(query, df):
    try:
        local_ns = {"df": df, "pd": pd, "np": np}
        exec(query, {"__builtins__": __builtins__}, local_ns)
        result = local_ns.get("result")
        if result is None:
            return None, "Script did not assign a `result` variable."
        return (None, f"Empty result: {result!r}") if _is_empty(result) else (result, None)
    except Exception as e:
        return None, str(e)


def build_pipeline(api_key, df, dataset_name="dataset"):
    agents = get_agents(api_key)
    A      = agents

    async def _sql_batch(questions, schema, semantic_json, sample, feedback_map):
        payload = {"questions": questions, "semantic_model": semantic_json,
                   "schema": schema, "dataframe_sample": sample}
        if feedback_map:
            payload["retry_feedback"] = feedback_map
        result = await _call(A["sql"], json.dumps(payload), "sql")
        answers = []
        for ans in result.output.answers:
            if not ans.error:
                res, err = _execute(ans.query, df)
                if err: ans.error = err
                else:
                    actual = _format_result(res)
                    ans.actual_result  = actual
                    ans.result_summary = f"Result:\n{actual}"
            answers.append(ans)
        return answers

    # ── Nodes ──────────────────────────────────────────────────────────────────

    async def semantic_node(state):
        print("  [semantic] Building semantic model...")
        cache_key = hashlib.md5(f"{df.shape}|{list(df.columns)}|{df.head(3).to_csv()}".encode()).hexdigest()
        if cache_key in _SEMANTIC_CACHE:
            validated = _SEMANTIC_CACHE[cache_key]
        else:
            result    = await _call(A["semantic"], json.dumps({"dataset_name": dataset_name, "profile": profile_dataframe(df)}), "semantic")
            validated = validate_semantics(result.output, df)
            _SEMANTIC_CACHE[cache_key] = validated
        floor = question_floor(validated)
        print(f"  [semantic] Done. Domain: {validated.inferred_domain} | Q floor: {floor}")
        log = list(state.get("pipeline_log") or [])
        log.append(f"semantic → done (floor={floor})")
        return {"semantic_model": validated, "question_floor": floor, "pipeline_log": log,
                "sql_retry_count": 0, "insights_retry_count": 0,
                "refinement_feedback": None, "refinement_failed_questions": None,
                "questions": None, "answers": None, "insights": None}

    async def question_node(state):
        floor = state.get("question_floor") or 10
        print(f"  [questions] Generating questions — floor: {floor}...")
        payload = {"semantic_model": state["semantic_model"].model_dump(),
                   "dataframe_sample": df.head(5).to_dict(orient="records"),
                   "question_floor": floor}
        if state.get("refinement_feedback"):
            payload["refinement_feedback"] = state["refinement_feedback"]
        result = await _call(A["question"], json.dumps(payload), "questions")
        qs     = result.output
        deduped, removed = dedup_questions(qs.questions)
        if removed:
            print(f"  [questions] Deduped {removed} duplicate(s). {len(qs.questions)} → {len(deduped)}")
        qs = AnalyticalQuestionsOutput(dataset_understanding=qs.dataset_understanding, questions=deduped)
        print(f"  [questions] {len(qs.questions)} questions ready.")
        log = list(state.get("pipeline_log") or [])
        log.append(f"questions → {len(qs.questions)} (deduped {removed})")
        return {"questions": qs, "refinement_feedback": None, "pipeline_log": log}

    async def sql_node(state):
        questions  = state["questions"]
        semantic   = state["semantic_model"]
        sql_retry  = state.get("sql_retry_count") or 0
        failed_qs  = state.get("refinement_failed_questions") or []
        existing   = list(state.get("answers") or [])
        schema     = [{"name": c.name, "dtype": c.dtype, "role": c.role, "description": c.description, "unit": c.unit} for c in semantic.columns]
        sem_json   = semantic.model_dump()
        sample     = df.head(5).to_dict(orient="records")
        feedback   = state.get("refinement_feedback")

        to_run = failed_qs if failed_qs else sorted(
            [q.question for q in questions.questions],
            key=lambda q: (0 if {q2.question: q2.priority for q2 in questions.questions}.get(q) else 1)
        )
        print(f"  [sql] Running {len(to_run)} questions (attempt {sql_retry+1}, ⭐ priority-first)...")

        sem = asyncio.Semaphore(2)  # stay under Kimi's 3-request concurrency limit
        async def _sem_batch(batch, fb):
            async with sem:
                return await _sql_batch(batch, schema, sem_json, sample, fb)
        tasks = [
            _sem_batch(to_run[i:i+BATCH_SIZE],
                       {q: feedback for q in to_run[i:i+BATCH_SIZE]} if feedback else {})
            for i in range(0, len(to_run), BATCH_SIZE)
        ]
        results = await asyncio.gather(*tasks)
        all_answers = [ans for batch in results for ans in batch]

        # Self-heal bad queries
        for heal in range(1, SQL_HEAL_RETRIES + 1):
            bad = [a for a in all_answers if a.error]
            if not bad: break
            print(f"  [sql] Self-healing {len(bad)} failed queries (pass {heal})...")
            fb_map = {a.question: f"Previous query: {a.query!r}\nError: {a.error}\nFix: single eval()-able pandas expression." for a in bad}
            healed = []
            heal_sem = asyncio.Semaphore(2)
            async def _sem_heal(batch):
                async with heal_sem:
                    return await _sql_batch([a.question for a in batch], schema, sem_json, sample,
                                           {a.question: fb_map[a.question] for a in batch})
            heal_tasks = [_sem_heal(bad[i:i+BATCH_SIZE]) for i in range(0, len(bad), BATCH_SIZE)]
            heal_results = await asyncio.gather(*heal_tasks)
            healed = [ans for batch in heal_results for ans in batch]
            healed_map  = {a.question: a for a in healed}
            all_answers = [healed_map.get(a.question, a) if a.error else a for a in all_answers]

        if failed_qs and existing:
            failed_set   = set(failed_qs)
            all_answers  = [a for a in existing if a.question not in failed_set] + all_answers

        good = sum(1 for a in all_answers if not a.error)
        print(f"  [sql] Done — {good}/{len(all_answers)} succeeded.")
        log = list(state.get("pipeline_log") or [])
        log.append(f"sql attempt {sql_retry+1} → {good}/{len(all_answers)} ok")
        return {"answers": all_answers, "refinement_feedback": None, "refinement_failed_questions": None, "pipeline_log": log}

    async def check_sql_node(state):
        retry = state.get("sql_retry_count") or 0
        print(f"  [check:sql] Checking results (budget left: {MAX_RETRIES - retry})...")
        answers_data = [{"question": a.question, "query": a.query,
                         "actual_result": a.actual_result if not a.error else None,
                         "error": a.error,
                         "priority": next((q.priority for q in state["questions"].questions if q.question == a.question), False)}
                        for a in state["answers"]]
        decision = (await _call(A["sql_checker"], json.dumps({"semantic_model": state["semantic_model"].model_dump(), "answers": answers_data}), "sql_checker")).output
        log = list(state.get("pipeline_log") or [])
        if decision.approved or retry >= MAX_RETRIES:
            status = "approved" if decision.approved else "budget exhausted"
            print(f"  [check:sql] ✅ {status}")
            log.append(f"check:sql → {status}")
            return {"_next": "insights_node", "refinement_feedback": None, "refinement_failed_questions": None, "pipeline_log": log}
        else:
            print(f"  [check:sql] ↺ Rejected: {decision.feedback}")
            log.append(f"check:sql → rejected (attempt {retry+1})")
            return {"_next": "sql_node", "refinement_feedback": decision.feedback,
                    "refinement_failed_questions": decision.failed_questions,
                    "sql_retry_count": retry + 1, "insights": None, "pipeline_log": log}

    async def insights_node(state):
        retry = state.get("insights_retry_count") or 0
        print(f"  [insights] Generating report (attempt {retry+1})...")
        qa_pairs = [{"question": a.question, "actual_result": a.actual_result if not a.error else None, "explanation": a.explanation, "error": a.error} for a in state["answers"]]
        payload  = {"semantic_model": state["semantic_model"].model_dump(),
                    "dataset_understanding": state["questions"].dataset_understanding,
                    "qa_pairs": qa_pairs}
        if state.get("refinement_feedback"):
            payload["refinement_feedback"] = state["refinement_feedback"]
        result = await _call(A["insights"], json.dumps(payload), "insights")
        ins = result.output
        deduped, removed = dedup_actions(ins.action_items)
        if removed:
            print(f"  [insights] Merged {removed} duplicate action item(s).")
        ins = BusinessInsightsOutput(executive_summary=ins.executive_summary,
                                     positive_highlights=ins.positive_highlights,
                                     action_items=deduped,
                                     watch_out_for=ins.watch_out_for)
        log = list(state.get("pipeline_log") or [])
        log.append(f"insights attempt {retry+1} → done (deduped {removed})")
        return {"insights": ins, "refinement_feedback": None, "pipeline_log": log}

    async def check_insights_node(state):
        retry = state.get("insights_retry_count") or 0
        print(f"  [check:insights] Checking report (budget left: {MAX_RETRIES - retry})...")
        ins     = state["insights"]
        prompt  = json.dumps({"semantic_model": state["semantic_model"].model_dump(),
                              "qa_pairs": [{"question": a.question, "actual_result": a.actual_result if not a.error else None, "error": a.error} for a in state["answers"]],
                              "insights": {"executive_summary": ins.executive_summary, "positive_highlights": ins.positive_highlights,
                                           "action_items": [a.__dict__ for a in ins.action_items], "watch_out_for": ins.watch_out_for}})
        decision = (await _call(A["insights_checker"], prompt, "ins_checker")).output
        log      = list(state.get("pipeline_log") or [])
        if decision.approved or retry >= MAX_RETRIES:
            status = "approved" if decision.approved else "budget exhausted"
            print(f"  [check:insights] ✅ {status}")
            log.append(f"check:insights → {status}")
            return {"_next": "done", "refinement_feedback": None, "pipeline_log": log}
        else:
            print(f"  [check:insights] ↺ Rejected: {decision.feedback}")
            log.append(f"check:insights → rejected (attempt {retry+1})")
            return {"_next": "insights_node", "refinement_feedback": decision.feedback,
                    "insights_retry_count": retry + 1, "insights": None, "pipeline_log": log}

    # ── Graph wiring ────────────────────────────────────────────────────────────
    def route_sql(s): return s.get("_next", "insights_node")
    def route_ins(s): return "__end__" if s.get("_next") == "done" else s.get("_next", "insights_node")

    g = StateGraph(MasterState)
    for name, fn in [("semantic_node", semantic_node), ("question_node", question_node),
                     ("sql_node", sql_node), ("check_sql", check_sql_node),
                     ("insights_node", insights_node), ("check_insights", check_insights_node)]:
        g.add_node(name, fn)

    g.set_entry_point("semantic_node")
    g.add_edge("semantic_node", "question_node")
    g.add_edge("question_node", "sql_node")
    g.add_edge("sql_node",      "check_sql")
    g.add_edge("insights_node", "check_insights")
    g.add_conditional_edges("check_sql",      route_sql, {"sql_node": "sql_node",       "insights_node": "insights_node"})
    g.add_conditional_edges("check_insights", route_ins, {"insights_node": "insights_node", "__end__": "__end__"})
    return g.compile()


def run_analysis(df, api_key, dataset_name="dataset"):
    async def _run():
        graph = build_pipeline(api_key, df, dataset_name)
        return await graph.ainvoke({
            "semantic_model": None, "questions": None, "answers": None, "insights": None,
            "question_floor": 10, "sql_retry_count": 0,
            "insights_retry_count": 0, "refinement_feedback": None,
            "refinement_failed_questions": None, "_next": "", "pipeline_log": [],
        })
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    try:
        return loop.run_until_complete(_run())
    finally:
        loop.close()

print("✅ Pipeline ready.")

✅ Pipeline ready.


## 9 · Load Data

In [10]:
# ── ✏️ Edit these two values ───────────────────────────────────────────────
KIMI_API_KEY = "sk-crXBPzT9OKn4kZRchU2n02cilSiPPQTDpfoZAcnZQYSj3wvI"
CSV_PATH       = "/kaggle/input/datasets/omaralnajjarr/financial-data/jordan_transactions.csv"
# ──────────────────────────────────────────────────────────────────────────

df = pd.read_csv(CSV_PATH)

# Derive a clean dataset name from the file path (used in the semantic model and report)
DATASET_NAME = os.path.splitext(os.path.basename(CSV_PATH))[0].replace("_", " ").replace("-", " ")

print(f"Loaded   : {len(df):,} rows x {len(df.columns)} columns")
print(f"Dataset  : {DATASET_NAME}")
print(f"Missing  : {df.isnull().sum().sum():,} values")
print(f"Memory   : {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
df.head(10)

Loaded   : 1,752 rows x 8 columns
Dataset  : jordan transactions
Missing  : 0 values
Memory   : 645.9 KB


,transaction_id,mall_name,branch_name,transaction_date,tax_amount,transaction_amount,transaction_type,transaction_status
0,JO-2504-4466-34760,Z Mall,Z Mall Al Bayader,20/04/2025 20:55,0.1520,2.0520,Sale,Failed
1,JO-2504-8444-31356,C Mall,C Mall Amman,20/04/2025 20:37,0.1520,2.0520,Sale,Failed
2,JO-2504-3064-71649,Z Mall,Z Mall Gardens,20/04/2025 20:09,0.1520,2.0520,Sale,Failed
3,JO-2504-2507-64030,Y Mall,Y Mall Tla'a Al-Ali,17/04/2025 7:03,0.1520,2.0520,Sale,Failed
4,JO-2504-1155-35357,C Mall,C Mall Irbid,13/04/2025 6:38,0.0700,1.1200,Sale,Failed
5,JO-2504-9356-34850,Z Mall,Z Mall Al Jubeiha,12/04/2025 20:04,0.0035,0.0535,Sale,Failed
6,JO-2504-5116-65058,Z Mall,Z Mall Al Bayader,29/03/2025 21:17,0.0600,0.8000,Sale,Failed
7,JO-2504-8072-48225,Y Mall,Y Mall Shmeisani,25/02/2025 10:19,0.2400,3.2700,Sale,Failed
8,JO-2504-7586-81795,Y Mall,Y Mall Tla'a Al-Ali,16/02/2025 19:23,0.3800,5.1100,Sale,Failed
9,JO-2504-6847-49531,Z Mall,Z Mall Al Jubeiha,16/02/2025 19:15,1.5000,20.3000,Sale,Failed


## 10 · Run Analysis

> ⏱ Typical runtime: 4–10 minutes

In [11]:
_AGENTS_CACHE.clear()
_SEMANTIC_CACHE.clear()

result = run_analysis(df, KIMI_API_KEY, dataset_name=DATASET_NAME)
print("\n✅ Analysis complete!")
print("\n── Pipeline log ──")
for entry in result.get("pipeline_log") or []:
    print(" ", entry)

  ✅ Agents ready (model=kimi-k2.5)
  [semantic] Building semantic model...
  [semantic] Done. Domain: retail financial transactions | Q floor: 26
  [questions] Generating questions — floor: 26...
  [questions] 33 questions ready.
  [sql] Running 33 questions (attempt 1, ⭐ priority-first)...


<string>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
<string>:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


  [sql] Self-healing 2 failed queries (pass 1)...
  [sql] Self-healing 2 failed queries (pass 2)...
  [sql] Done — 31/33 succeeded.
  [check:sql] Checking results (budget left: 2)...
  [check:sql] ✅ approved
  [insights] Generating report (attempt 1)...
  [check:insights] Checking report (budget left: 2)...
  [check:insights] ✅ approved

✅ Analysis complete!

── Pipeline log ──
  semantic → done (floor=26)
  questions → 33 (deduped 0)
  sql attempt 1 → 31/33 ok
  check:sql → approved
  insights attempt 1 → done (deduped 0)
  check:insights → approved


## 11 · Results

In [12]:
semantic_model = result.get("semantic_model")

if semantic_model:
    q_floor = result.get("question_floor") or "—"
    print(f"Dataset      : {semantic_model.dataset_name}")
    print(f"Domain       : {semantic_model.inferred_domain or '—'}")
    print(f"Rows         : {semantic_model.row_count:,}")
    print(f"Question floor (adaptive) : {q_floor}")
    if semantic_model.primary_keys:
        print(f"PKs          : {', '.join(semantic_model.primary_keys)}")
    if semantic_model.time_column:
        print(f"Time col     : {semantic_model.time_column}")
    if semantic_model.date_range:
        dr = semantic_model.date_range
        print(f"Date range   : {dr.get('min')} → {dr.get('max')} ({dr.get('span_days')} days)")
    print()

    schema_rows = [
        {"Column": c.name, "Type": c.dtype, "Role": c.role,
         "Domain": c.business_domain or '—',
         "Cardinality": c.cardinality or '—',
         "Description": c.description, "Unit": c.unit or "—"}
        for c in semantic_model.columns
    ]
    display(pd.DataFrame(schema_rows))

    if semantic_model.inter_column_relationships:
        print("\nColumn relationships:")
        for rel in semantic_model.inter_column_relationships:
            print(f"  • {rel}")

    missing_nonzero = {k: v for k, v in semantic_model.missing_values.items() if v > 0}
    print("\n" + ("\u2705 No missing values." if not missing_nonzero else "Missing values:"))
    if missing_nonzero:
        display(pd.DataFrame([{"Column": k, "Missing": v} for k, v in missing_nonzero.items()]))
else:
    print("No semantic model available.")

Dataset      : jordan transactions
Domain       : retail financial transactions
Rows         : 1,752
Question floor (adaptive) : 26
PKs          : transaction_id
Time col     : transaction_date
Date range   : None → None (None days)



,Column,Type,Role,Domain,Cardinality,Description,Unit
0,transaction_id,string,id,identifier,unique,Unique identifier for each transaction followi...,—
1,mall_name,category,category,geographic,low,Name of the shopping mall where the transactio...,—
2,branch_name,category,category,geographic,low,"Specific branch location within a mall chain, ...",—
3,transaction_date,datetime,time,temporal,high,Date and time when the transaction was recorde...,—
4,tax_amount,float,feature,financial,high,"Sales tax amount applied to the transaction, c...",JOD
5,transaction_amount,float,feature,financial,high,Total monetary value of the transaction before...,JOD
6,transaction_type,category,category,operational,low,Classification of transaction as either a stan...,—
7,transaction_status,category,category,status,low,Processing status indicating whether the trans...,—



Column relationships:
  • transaction_id is the primary key uniquely identifying each record
  • branch_name is hierarchically related to mall_name (each branch belongs to one mall)
  • transaction_date provides temporal ordering for all transactions
  • tax_amount has perfect positive correlation (r=1.0) with transaction_amount indicating tax is a fixed percentage of transaction amount
  • mall_name and branch_name form a geographic hierarchy
  • transaction_type and transaction_status together define transaction outcome categories

✅ No missing values.


In [13]:
questions_output = result.get("questions")
q_removed = result.get("questions_removed_count") or 0

if questions_output:
    print(f"Dataset understanding:\n{questions_output.dataset_understanding}\n")
    print(f"{len(questions_output.questions)} questions after deduplication", end="")
    if q_removed:
        print(f" ({q_removed} duplicate(s) removed \u2705)")
    else:
        print()

    by_category = defaultdict(list)
    for q in questions_output.questions:
        by_category[q.category].append(q)

    priority_count = sum(1 for q in questions_output.questions if q.priority)
    print(f"\u2b50 {priority_count} high-priority questions (ran in batch 1)\n")
    print("=" * 70)

    for category, qs in by_category.items():
        print(f"\n\u25b6 {category} ({len(qs)} questions)")
        print("-" * 50)
        for q in qs:
            marker = "\u2b50" if q.priority else " •"
            print(f"  {marker} {q.question}")
            if q.priority and q.priority_reason:
                print(f"      \u2192 {q.priority_reason}")
else:
    print("No questions available.")

Dataset understanding:
This is a retail transaction dataset from Jordanian shopping malls containing 1,752 transactions across 3 mall chains (C Mall, Z Mall, Y Mall) and 9 branches spanning from January to April 2025. The dataset captures financial transactions with amounts in Jordanian Dinar (JOD), including tax calculations. Key characteristics: 93.4% completion rate, 99.4% sales vs 0.6% refunds, perfect correlation between tax_amount and transaction_amount (fixed tax rate), hierarchical geographic structure (mall → branch), and temporal coverage over ~3 months. The data shows right-skewed transaction amounts (mean 8.36 JOD, median 6.55 JOD) with a long tail up to 85.52 JOD.

33 questions after deduplication
⭐ 21 high-priority questions (ran in batch 1)


▶ Distribution Analysis (4 questions)
--------------------------------------------------
  ⭐ What is the distribution of transaction amounts across all transactions, and what percentage fall into low-value (under 5 JOD), medium-valu

In [14]:
answers = result.get("answers") or []

if answers:
    good = [a for a in answers if not a.error]
    bad  = [a for a in answers if a.error]
    print(f"Total questions : {len(answers)}")
    print(f"Successful      : {len(good)}")
    print(f"Failed          : {len(bad)}")
    print("=" * 70)

    if good:
        print("\n\u2705 SUCCESSFUL QUERIES\n")
        for i, a in enumerate(good, 1):
            print(f"[{i}] {a.question}")
            print(f"    Query   : {a.query}")
            print(f"    Result  : {(a.actual_result or '')[:300]}")
            print(f"    Context : {a.explanation}")
            print()

    if bad:
        print("\n\u274c FAILED QUERIES\n")
        for a in bad:
            print(f"  Q: {a.question}")
            print(f"  Error: {a.error}")
            if a.query:
                print(f"  Query: {a.query}")
            print()
else:
    print("No query results available.")

Total questions : 33
Successful      : 31
Failed          : 2

✅ SUCCESSFUL QUERIES

[1] What is the distribution of transaction amounts across all transactions, and what percentage fall into low-value (under 5 JOD), medium-value (5-15 JOD), and high-value (over 15 JOD) tiers?
    Query   : # Define value tiers based on transaction amount
total_transactions = len(df)

low_value = df[df['transaction_amount'] < 5]
medium_value = df[(df['transaction_amount'] >= 5) & (df['transaction_amount'] <= 15)]
high_value = df[df['transaction_amount'] > 15]

low_count = len(low_value)
medium_count = len(medium_value)
high_count = len(high_value)

low_pct = (low_count / total_transactions) * 100
medium_pct = (medium_count / total_transactions) * 100
high_pct = (high_count / total_transactions) * 100

result = {
    'total_transactions': total_transactions,
    'low_value': {'count': low_count, 'percentage': low_pct, 'range': '< 5 JOD'},
    'medium_value': {'count': medium_count, 'percentage': medium_

In [15]:
insights = result.get("insights")
i_removed = result.get("insights_removed_count") or 0

if insights:
    if i_removed:
        display(HTML(f"<p style='color:#888;font-size:0.85rem'>\u2705 {i_removed} near-duplicate action item(s) merged by deduplication</p>"))

    display(HTML("<h3>\U0001f4cb Executive Summary</h3>"))
    display(HTML(f"<blockquote style='background:#f0f4ff;padding:12px 16px;border-left:4px solid #4a6cf7;border-radius:6px'>{insights.executive_summary}</blockquote>"))

    if getattr(insights, "positive_highlights", None):
        display(HTML("<h3>\u2705 What\'s Working Well</h3>"))
        highlights_html = "<div style=\'background:#e8f5e9;border-left:4px solid #38a169;padding:14px 18px;border-radius:6px;margin-bottom:12px\'>"
        highlights_html += "<ul style=\'margin:0;padding-left:20px\'>"
        for h in insights.positive_highlights:
            highlights_html += f"<li style=\'margin-bottom:8px;color:#1a6334\'><strong>\U0001f7e2</strong> {h}</li>"
        highlights_html += "</ul></div>"
        display(HTML(highlights_html))

    display(HTML("<h3>\U0001f3af Action Items</h3>"))
    priority_colors  = {"High": "#fde8e8", "Medium": "#fef3cd", "Low": "#e8f5e9"}
    priority_borders = {"High": "#e53e3e", "Medium": "#d69e2e", "Low": "#38a169"}
    priority_icons   = {"High": "\U0001f534", "Medium": "\U0001f7e1", "Low": "\U0001f7e2"}

    for action in insights.action_items:
        bg     = priority_colors.get(action.priority, "#f9f9f9")
        border = priority_borders.get(action.priority, "#999")
        icon   = priority_icons.get(action.priority, "")
        html = f"""
        <div style='background:{bg};border-left:4px solid {border};padding:14px 18px;border-radius:6px;margin-bottom:12px'>
          <div style='font-weight:700;font-size:1.05rem'>{icon} {action.title} &nbsp;<span style='font-weight:400;font-size:0.85rem;color:#555'>({action.priority} Priority)</span></div>
          <div style='margin-top:8px'><strong>What's happening:</strong> {action.what}</div>
          <div style='margin-top:4px'><strong>Why it matters:</strong> {action.why_it_matters}</div>
          <div style='margin-top:4px'><strong>Recommendation:</strong> {action.recommendation}</div>
          <div style='margin-top:4px'><strong>Expected impact:</strong> {action.expected_impact}</div>
        </div>
        """
        display(HTML(html))

    display(HTML("<h3>\u26a0\ufe0f Watch Out For</h3>"))
    for warning in insights.watch_out_for:
        display(HTML(f"<div style='background:#fff8e1;border-left:4px solid #f59e0b;padding:10px 14px;border-radius:5px;margin-bottom:8px'>\u26a0\ufe0f {warning}</div>"))
else:
    print("No insights available.")